# Ingenieria de Prompt con Langchain y uso de plantillas de prompt de LangChain

## Objetivos

* **Comprender los conceptos básicos de la ingeniería de prompts**: Obtener una base sólida sobre cómo comunicarse eficazmente con los LLM mediante prompts, preparando el terreno para técnicas más avanzadas.
* **Dominar técnicas avanzadas de prompts**: Aprender y aplicar métodos avanzados de ingeniería de prompts, como el aprendizaje de pocos ejemplos (few-shot) y el aprendizaje de consistencia propia (self-consistent learning), para optimizar las respuestas del LLM.
* **Utilizar plantillas de prompts de LangChain**: Adquirir fluidez en el uso de las plantillas de prompts de LangChain para estructurar y optimizar tus interacciones con los LLM.
* **Desarrollar agentes de LLM prácticos**: Adquirir las habilidades para crear e implementar agentes, como bots de preguntas y respuestas (QA) y herramientas de resumen de texto, utilizando las plantillas de prompts de LangChain, traduciendo el conocimiento teórico en soluciones prácticas.

## Setup

Para este laboratorio usamos el entorno de practicas creado en el articulo: Preparación del entorno RAG híbrido (Ollama + DeepSeek + LangChain) para prácticas.

### Importar librerias 

Importar variables del entorno y modulo LLMconfig

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

from llm_config import get_llm

## Configurar los LLM

En esta seccion configuramnos los LLM usando nuestro entorno hibrido **DeepSeek + Ollama** utilizando el modulo `llm_config`:

- `lmm` :  Especifica el modulo a utilizar. `sdk` para deepseek modelo `deepseek-chat` y `oll` para LLM local Ollama modelo `qwen:4b`. Por defecto utiliza DeepSeek. 
- `params` :  Configura los parametros del modelo establecidos por defecto como: 
    - `temperature`: 1, Aleatoriedad del modelo.
    - `max_tokens`: 50, Longitud de la respuesta en DeepSeek
    - `top_p`: 1, Diversidad de palabras considerando la probabilidad acumulada (no usar junto con temperature).
    - `top_k`: 40, Diversidad de palabras considerando la frecuencia 
    - `frequency_penalty`: 0, Penalizacion de frecuencia - DeepSeek
    - `presence_penalty`: 0, Penalizacion de presencia - DeepSeek
    - `repeat_penalty`: 1.1 Penalizacion de repeticion - Ollama

Para este lab utilizaremos solamente DeepSeek por ser un modelo mas grande y provee mejores respuestas. 

Ejemplo de llamada con `llmconfig`:
```python
get_llm(llm="dsk", params={"temperature": 0.8, "max_tokens": 30})
```

## Uso de LangChain para interactuar con el modelo

Cuando trabajamos con modelos de lenguaje en LangChain, el método principal para ejecutar el modelo es `.invoke()`. Es el punto central de interacción con el modelo y forma parte del diseño unificado de LangChain y muchos componentes implementan la misma interfaz.

LangChain hace internamente:

1. Prepara la entrada
2. Llama al backend (DeepSeek / Ollama / etc.)
3. Recibe la respuesta

Ejemplo de uso de `.invoke()`: 
```python
llm = get_llm(llm="dsk", params={"temperature": 0.8, "max_tokens": 30})
response = llm.invoke("Explícame qué es PCA")
print(response)
```

## Estructura de la respuesta de los LLM

La respuesta no siempre es un simple texto. En muchos casos (especialmente con Chat Models como DeepSeek), obtienes un objeto estructurado con mucha más información que pueden ser extraidas como objetos generados por el modelo que estes usando. 

EL objeto `content`, es el que tiene la respuesta del prompt, para que se muestre solo ese usamos `response.content`. Este viene formateado en Markdown. Para vizualizar correctamente el formato importamos librerias de vizualizacion de jupyter: 

In [ ]:
from IPython.display import display, Markdown

Definimos una funcion para reemplazar a `print()` en la respuesta que muestre el texto en el formato correcto: 

In [ ]:
def show(response):
    content = response.content if hasattr(response, "content") else response
    display(Markdown(content))

Ahora solamante llamamos a show(response) y devolvera `content` vizualizado correctamnente. 

Ejemplo de la respuesta completa de DeepSeek en el ejemplo siguiente:  

```text
 content " ..."
 additional_kwargs={'refusal': None} 
 response_metadata={'token_usage': {
    'completion_tokens': 957, 
    'prompt_tokens': 14, 
    'total_tokens': 971, 
    'completion_tokens_details': None, 
    'prompt_tokens_details': {
        'audio_tokens': None, 
        'cached_tokens': 0}, 
    'prompt_cache_hit_tokens': 0, 
    'prompt_cache_miss_tokens': 14
    }, 
'model_provider': 'openai', 
'model_name': 'deepseek-v4-flash', 
'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 
'id': '8eb89977-19f8-4756-b53d-4f721d92fcef', 
'finish_reason': 'stop', 
'logprobs': None} 
id='lc_run--019dd8b6-ff91-75f1-b656-aa3691947150-0' tool_calls=[] invalid_tool_calls=[] 
usage_metadata={
    'input_tokens': 14, 
    'output_tokens': 957, 
    'total_tokens': 971, 
    'input_token_details': {
        'cache_read': 0}, 
    'output_token_details': {}}
```


### Prompt Basico: 

In [ ]:
# invocar el modelo + paranetros
llm = get_llm(llm="dsk", params={"temperature": 0.8})

# prompt
prompt = "Cual es un buen nombre para un perro"

response = llm.invoke(prompt)
print(f"prompt: {prompt}\n")
show(response)

prompt: Cual es un buen nombre para un perro



'¡Elegir el nombre perfecto para un perro es una decisión muy especial! Depende mucho de la personalidad del perro, su raza, tamaño y, por supuesto, de tus gustos personales.\n\nPara ayudarte, aquí te dejo una lista organizada por categorías con ideas de nombres populares y originales:\n\n### 1. Clásicos y Populares (Siempre funcionan)\nSon fáciles de pronunciar, cortos y a los perros les resultan fáciles de reconocer.\n- **Machos:** Max, Toby, Rocky, Simba, Leo, Bruno, Coco, Milo, Zeus, Thor.\n- **Hembras:** Luna, Nina, Lola, Maya, Kira, Chloe, Bella, Dora, Nala, Kiara.\n\n### 2. Originales y Únicos (Para un perro con mucha personalidad)\n- **Inspirados en la Mitología:** Odín, Apolo, Atenea, Hera, Loki, Artemisa, Hades.\n- **Inspirados en la Naturaleza:** Río, Sol, Nube, Bosque, Brisa, Trueno, Nieve, Otoño.\n- **Inspirados en la Gastronomía (muy tiernos):** Canela, Mochi, Brownie, Waffle, Pan, Kiwi, Café, Wasabi.\n- **Inspirados en Viajes:** Kyoto, Alaska, Cairo, Venecia, Koda, París